# Gemma 4: Task Soundness, Serving and Evaluation

## Summary

Comparing coding agents locally is useful only if three things hold: the tasks can actually be checked, the model is served the same way as in the competition, and the agent fits the twelve-hour limit. This notebook measures each of them on the competition's 129 public tasks with the official serving stack.

- **Tasks.** 114 of the 129 public tasks are valid for local evaluation: their tests fail on the unchanged repository and pass with the reference fix. The other 15 are excluded, each with a recorded reason.
- **Agents.** On 70 tasks from FastAPI, Requests and HTTPX, a single agent with four tools solved 27, against 16 for a two-agent reference. On 44 tasks from a fourth repository, Rich, both solved 12, but the single agent needed 43% less time per task.
- **Time.** About 120 tasks in twelve hours leave roughly six minutes per task, so an eight-minute limit works only when most tasks finish earlier.
- **Serving and adapters.** An early serving build turned the agent's history into a different prompt from the one intended, and a short LoRA adaptation lowered the training loss but solved fewer tasks.

The submitted archive is the unchanged public Budget-Fit agent, the configuration that scored 0.18. Since the first release the attached datasets have grown: a comparison of the 31B, 12B and E4B models, 165 extra verified tasks for local validation, and 1,670 agent trajectories for LoRA training. [Task Soundness and Local Results](https://www.kaggle.com/datasets/dmitriigluzdov/gemma-4-task-soundness-and-local-results), [Serving and LoRA Verification Kit](https://www.kaggle.com/datasets/dmitriigluzdov/gemma-4-serving-and-lora-verification-kit).

In [ ]:
from pathlib import Path
from io import StringIO
import hashlib, json, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
WORK=Path('/kaggle/working');WORK.mkdir(parents=True,exist_ok=True)
INPUT=Path('/kaggle/input')
def locate(name):
    paths=sorted(INPUT.rglob(name))
    assert len(paths)==1, (name,len(paths))
    return paths[0].parent
A=locate('task_soundness.csv');K=locate('gate_measurements.csv')
for folder in (A,K):
    for record in json.loads((folder/'manifest.json').read_text())['files']:
        assert hashlib.sha256((folder/record['name']).read_bytes()).hexdigest()==record['sha256']
def evidence(folder,name):
    return (folder/name).read_text(encoding='utf8')
def frame(folder,name):return pd.read_csv(StringIO(evidence(folder,name)))
audit=frame(A,'task_soundness.csv');outcomes=frame(A,'all_config_eval.csv')
summary=frame(A,'config_ablation_summary.csv');coverage=frame(A,'task_coverage.csv')
pairs=frame(A,'paired_comparisons.csv');runtime=frame(A,'time_budget_calculator.csv')
lb=frame(A,'leaderboard_observations.csv');protocol=json.loads(evidence(A,'evaluation_protocol.json'))
render=frame(K,'served_prompt_checks.csv');gate=frame(K,'gate_measurements.csv').set_index('measurement').value
loss=frame(K,'adaptation_loss_changes.csv');adaptation=json.loads(evidence(K,'adaptation_evaluation.json'))
registry=json.loads(evidence(K,'archive_registry.json'))
budget_summary=frame(A,'runtime_budget_summary.csv');thinking_screen=frame(A,'public_thinking_screen.csv')
catalogue=frame(K,'public_bundle_catalogue.csv')
assert thinking_screen.set_index('thinking_budget_tokens').resolved.to_dict()=={2048:50,4096:48}
assert len(audit)==129 and audit.task_id.is_unique and audit.sound.sum()==114
assert len(outcomes)==692 and not outcomes.duplicated(['config','split','task_id']).any()
assert not outcomes.isna().any().any() and outcomes.resolved.isin([0,1]).all()
assert set(outcomes.task_id)==set(audit.loc[audit.sound==1,'task_id'])
assert not set(audit.query("split == 'development'").repo) & set(audit.query("split == 'held_out'").repo)
SINGLE='Four-tool single coder';REFERENCE='Rozen coder–analyzer reference';SCHEDULE='Six-minute budget schedule'
BG,INK,MUTED,EDGE='#0b1424','#eef3fa','#9fb0c4','#33425a'
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.spines.top':False,'axes.spines.right':False,'axes.titleweight':'bold','axes.titlelocation':'left','text.color':INK,'axes.labelcolor':MUTED,'axes.edgecolor':EDGE,'xtick.color':MUTED,'ytick.color':INK,'axes.facecolor':BG,'figure.facecolor':BG,'savefig.facecolor':BG,'legend.labelcolor':INK})
TEAL,BLUE,AMBER,RED,GRAY='#7fd8ff','#3d7bff','#a77bff','#ff6f91','#3a4a63'
def save(fig,name,caption,source):
    fig.text(.02,.057,caption,ha='left',fontsize=10)
    fig.text(.02,.008,'Source: '+source,ha='left',fontsize=9,color=MUTED)
    fig.savefig(WORK/name,dpi=160,bbox_inches='tight');plt.show();plt.close(fig)
def table(df):
    headers=list(df.columns)
    body='| '+' | '.join(headers)+' |\n| '+' | '.join(['---']*len(headers))+' |\n'
    body+='\n'.join('| '+' | '.join(str(v) for v in row)+' |' for row in df.itertuples(index=False,name=None))
    display(Markdown(body))

In [ ]:
fig,axs=plt.subplots(1,3,figsize=(12.8,3.6))
rich=summary.query("split == 'held_out'").set_index('config')
saving=100*(1-rich.loc[SINGLE,'mean_seconds']/rich.loc[REFERENCE,'mean_seconds'])
cards=[('VALID TASKS',f'{audit.sound.sum()} / {len(audit)} tasks','Empty patch fails, reference fix passes\nNo error in either run',TEAL),('AGENT COMPARISON',f'{len(outcomes)} outcomes','Same tasks for every agent\n70 development + 44 Rich tasks',BLUE),('QUALITY AND TIME',f'{saving:.1f}% less time','Single coder and reference: 12/44 each on Rich\nMean time per task: 216 vs 381 s',AMBER)]
for ax,(title,value,detail,color) in zip(axs,cards):
    ax.axis('off');ax.text(.02,.93,title,weight='bold',fontsize=12,color=color,transform=ax.transAxes)
    ax.text(.02,.61,value,fontsize=23,weight='bold',transform=ax.transAxes)
    ax.text(.02,.27,detail,fontsize=11,linespacing=1.7,transform=ax.transAxes)
    ax.plot([.02,.95],[.08,.08],color=color,lw=4,transform=ax.transAxes)
fig.subplots_adjust(left=.03,right=.99,top=.91,bottom=.26,wspace=.15)
save(fig,'01_graphical_abstract.png','Figure 1. Valid tasks, paired agent outcomes and time per task are separate checks.','task_soundness.csv; all_config_eval.csv; config_ablation_summary.csv')

*Figure 1. Summary of the three checks. Counts concern public tasks; the time saving is a local comparison on Rich, not a forecast for the competition hardware.*

## 1. Tasks, repositories and terms

Each task gives the agent an issue report and a snapshot of a Python repository. The agent edits the code, and its patch is checked with the task's tests, as in [SWE-bench](https://arxiv.org/abs/2310.06770). The competition's public tasks come from four repositories: FastAPI (67 tasks), [Rich](https://github.com/Textualize/rich), a library for formatted terminal output (48), Requests (13) and HTTPX (1). To avoid choosing and testing an agent on the same code, FastAPI, Requests and HTTPX form the **development set** used to compare agents, and Rich is kept as a separate **confirmation set**. Rich had been looked at during earlier work, so it is a check on a different repository rather than a fully unseen test.

Terms used below:

- **valid task** — the tests fail on the unchanged repository and pass with the reference fix, with no error in either run; the dataset and the title call such tasks *sound*;
- **paired comparison** — two agents on exactly the same tasks; only the tasks solved by one agent and not the other (its wins and losses) decide the difference;
- **agent plus setup time** — the time an agent spends on a task, including preparing its sandbox but not the final grading.

All runs use the permitted `gemma-4-31b-it-qat-w4a16-ct` model. Every number comes from the two attached datasets: [Task Soundness and Local Results](https://www.kaggle.com/datasets/dmitriigluzdov/gemma-4-task-soundness-and-local-results) and [Serving and LoRA Verification Kit](https://www.kaggle.com/datasets/dmitriigluzdov/gemma-4-serving-and-lora-verification-kit).

## 2. How the measurements were made

### 2.1 Task check

Every public task was run twice without an agent: once with an empty patch and once with the reference patch. A task is kept when the empty patch fails, the reference patch passes and neither run reports an error; it is also excluded when the local environment would skip one of its required tests. Nine environments needed a missing dependency (seven for Rich, two for FastAPI); tests, patches and grading were not changed. The list of valid tasks was fixed before any agent was compared.

### 2.2 Serving check

The agent framework sends the conversation history to the model server, which turns it into the prompt the model reads. If the server builds a different prompt from the one the agent intended, the model sees a different conversation. The check rebuilds the prompts from recorded requests and compares their lengths with what the server actually received, and the exact tokens for a few requests. Appendix A gives the result.

### 2.3 Agent comparison

Eight agent configurations ran on the same 70 development tasks, and three of them also on the 44 Rich tasks. The **reference** is Roman Rozen's two-agent design, in which a coder can consult a separate analyzer agent. The **single coder** is one agent with four tools, a five-minute limit and 20 tool calls. The other configurations change the reference's time limit, turn limit, time plan, search instruction or analyzer, or add two reviewers that vote on the patch, so only some pairs differ in a single setting. All runs used one A100 with eight tasks in parallel and the same request seed; a run that failed with an error counts as unsolved. Differences are tested with an exact sign test on wins and losses. Its probabilities are a rough guide only: tasks from one repository are not independent, and many comparisons were made.

### 2.4 Time budget

The total time of a run is approximately

$$T_{\mathrm{hours}}=\frac{S+N\,\bar{t}\,m}{3600},$$

where $S$ is the start-up time (900 seconds assumed), $N$ the number of tasks (about 120), $\bar{t}$ the mean agent plus setup time per task and $m$ a safety margin (1.25 in the main scenario). The [limit](https://www.kaggle.com/competitions/gemma-4-developer-agent/overview/evaluation) is twelve hours; waiting in the queue does not count, [according to the organizers](https://www.kaggle.com/competitions/gemma-4-developer-agent/discussion/747519). The scenario values are assumptions, not measurements of the competition hardware.

## 3. Results

### 3.1 Valid tasks

Of the 129 public tasks, the reference patch passed in 119 and the empty patch in two. After removing runs with errors and skipped required tests, **114 valid tasks** remain and 15 are excluded: FastAPI 62 of 67, Rich 44 of 48, Requests 7 of 13 and HTTPX 1 of 1. The development set therefore has 70 tasks and the confirmation set 44.

In [ ]:
order=['fastapi/fastapi','Textualize/rich','psf/requests','encode/httpx']
statuses=['sound','passes_without_fix','control_error','gold_does_not_pass_local_grader']
counts=pd.crosstab(audit.repo,audit.status).reindex(index=order,columns=statuses,fill_value=0)
fig,ax=plt.subplots(figsize=(11.5,4.8));left=np.zeros(len(order))
for status,label,color in zip(statuses,['Valid','Passes without a fix','Control error','Reference fails'],[TEAL,AMBER,RED,GRAY]):
    values=counts[status].to_numpy();ax.barh(order,values,left=left,label=label,color=color,height=.6)
    for i,v in enumerate(values):
        if v>=3:ax.text(left[i]+v/2,i,str(v),ha='center',va='center',color=BG if status=='sound' else INK,weight='bold')
    left+=values
for i,repo in enumerate(order):ax.text(left[i]+1,i,f"{counts.loc[repo,'sound']}/{int(left[i])} retained",va='center')
ax.set_xlim(0,85);ax.invert_yaxis();ax.set_xlabel('Public tasks (count)');ax.set_title('Task check results by repository')
ax.legend(ncol=4,loc='upper center',bbox_to_anchor=(.48,-.19),frameon=False,fontsize=10)
fig.subplots_adjust(left=.17,right=.98,top=.87,bottom=.33)
save(fig,'02_task_soundness.png','Figure 2. 114 of 129 public tasks are valid: the empty patch fails and the reference fix passes.','task_soundness.csv; dependency_repairs.csv')
quality=pd.DataFrame([{'Table':'Control audit','Rows':len(audit),'Duplicate keys':int(audit.task_id.duplicated().sum()),'Missing cells':int(audit.isna().sum().sum())},{'Table':'Paired outcomes','Rows':len(outcomes),'Duplicate keys':int(outcomes.duplicated(['config','split','task_id']).sum()),'Missing cells':int(outcomes.isna().sum().sum())}])
table(quality)
examples=audit.set_index('task_id').loc[['fastapi_11194','fastapi_14077','rich_3480'],['empty_resolved','gold_resolved','empty_exit_code','sound']].reset_index()
examples['Audit interpretation']=['Valid','Passes without a fix','Empty run times out']
examples.columns=['Task','Empty patch passes','Reference fix passes','Empty-run exit code','Kept','Meaning'];table(examples)

*Figure 2. Check results by repository. The tables below show that no rows are missing or duplicated, and three example tasks: a valid one, one that already passes without a fix, and one whose empty run timed out (exit code 124).*

### 3.2 Quality and time

**Table 1.** All configurations. "Not measured" means the configuration was not run on Rich. Times include sandbox setup and exclude grading; the 95th percentile describes slow tasks, not uncertainty.

In [ ]:
order=[REFERENCE,SINGLE,'Six-minute reference','Six-minute reference with 150 turns','Two-reviewer voting pipeline',SCHEDULE,'Source-first localization','Optional analyzer']
rows=[]
for config in order:
    d=summary.query("config == @config and split == 'development'").iloc[0]
    r=summary.query("config == @config and split == 'held_out'")
    rows.append({'Configuration':config,'Development':f'{d.resolved}/{d.n}','Rich':f'{r.iloc[0].resolved}/{r.iloc[0].n}' if len(r) else 'Not measured','Mean / p95 (s)':f'{d.mean_seconds:.0f} / {d.p95_seconds:.0f}','Session (min) / calls':f'{d.max_time_minutes} / {d.max_tool_calls}'})
table(pd.DataFrame(rows))
fig,(ax,bx)=plt.subplots(1,2,figsize=(12.5,5.1));x=np.arange(2);w=.32
for config,offset,color,label in [(SINGLE,-w/2,TEAL,'Single coder'),(REFERENCE,w/2,BLUE,'Reference (coder + analyzer)')]:
    sub=summary[summary.config==config].set_index('split');c=sub.loc[['development','held_out'],'resolved'].to_numpy();n=sub.loc[['development','held_out'],'n'].to_numpy()
    bars=ax.bar(x+offset,100*c/n,w,color=color,label=label);ax.bar_label(bars,labels=[f'{int(a)}/{int(b)}' for a,b in zip(c,n)],padding=4)
    r=sub.loc['held_out'];bars=bx.bar(x+offset,[r.mean_seconds,r.p95_seconds],w,color=color,label=label);bx.bar_label(bars,labels=[f'{r.mean_seconds:.0f}',f'{r.p95_seconds:.0f}'],padding=4)
ax.set_xticks(x,['Development','Rich (confirmation)']);ax.set_ylim(0,53);ax.set_ylabel('Resolved tasks (%)');ax.set_title('The development gain disappears on Rich');ax.legend(frameon=False,fontsize=10)
bx.set_xticks(x,['Mean','95th percentile']);bx.set_ylim(0,1050);bx.set_ylabel('Agent + setup time per task (s)');bx.set_title('The single coder is faster on Rich');bx.legend(frameon=False,fontsize=10)
fig.subplots_adjust(left=.07,right=.99,top=.87,bottom=.24,wspace=.31)
save(fig,'03_paired_quality_time.png','Figure 3. Same Rich score, 43.4% less time per task for the single coder.','all_config_eval.csv; config_ablation_summary.csv')

*Figure 3. On development tasks the single coder solved 27 of 70 against 16 for the reference (14 wins, 3 losses; sign test p = 0.013). On Rich both solved 12 of 44 (5 wins, 5 losses), while the single coder's mean time per task was 216 seconds against 381. Equal scores on 44 tasks do not prove equal quality.*

The six-minute time plan, chosen because it did well on development tasks, solved only **7 of 44** Rich tasks against 12 for the reference (1 win, 6 losses). Two voting reviewers solved **13 of 70** development tasks against 16 (3 wins, 6 losses). Neither is better than the reference.

Most tasks do not separate the agents. Of the development tasks, 37 were solved by no configuration and 5 by all eight, leaving 28 that differ; on Rich, 27 were solved by none, 5 by all three and 12 by some.

### 3.3 Time budget

With 120 tasks and 15 minutes of start-up, twelve hours allow **352 seconds per task**, or **282 seconds** with a 25% margin. If every task used a full eight-minute limit, the run would need 16 hours before start-up, and each extra minute per task adds two hours. A long per-task limit is safe only when most tasks finish earlier.

In [ ]:
assumptions=protocol['runtime_assumptions'];N=assumptions['task_count'];S=assumptions['startup_seconds']
limit=assumptions['limit_hours'];margin=assumptions['safety_factor'];allowed=(limit*3600-S)/(N*margin)
assert abs(allowed-282)<1e-9
fig,ax=plt.subplots(figsize=(10.5,5.2));seconds=np.linspace(150,500,101)
for m,color,label in [(1,BLUE,'No task-time margin'),(margin,AMBER,'25% task-time margin')]:ax.plot(seconds,(S+N*seconds*m)/3600,color=color,lw=2.3,label=label)
ax.axhline(limit,color=RED,ls='--',label='12-hour limit');ax.axvline(allowed,color=GRAY,ls=':')
ax.text(allowed-8,4.9,'282 s/task',rotation=90,ha='right',va='bottom',fontsize=9,color=MUTED)
ax.set_xlim(150,500);ax.set_ylim(4,22);ax.set_xlabel('Assumed mean agent + setup time per task (s)');ax.set_ylabel('Projected total time (h)')
ax.set_title('120 tasks, 15-minute start-up');ax.legend(frameon=False,loc='upper left')
fig.subplots_adjust(left=.09,right=.98,top=.87,bottom=.24)
save(fig,'04_runtime_sensitivity.png','Figure 4. The twelve-hour limit constrains the mean time per task, not only the per-task limit.','time_budget_calculator.csv; evaluation_protocol.json')
rt=budget_summary.copy()
def measured(x):return 'Not measured' if pd.isna(x) else f'{x:.2f}'
rt['Mean / p95 (s)']=[f'{a:.2f} / {b:.2f}' if pd.notna(a) else 'Not measured' for a,b in zip(rt.mean_seconds,rt.p95_seconds)]
for field in ['total_hours_mean','headroom_hours_mean','total_hours_at_p95']:rt[field]=rt[field].map(measured)
rt=rt[['configuration','Mean / p95 (s)','total_hours_mean','headroom_hours_mean','total_hours_at_p95']]
rt.columns=['Configuration','Mean / p95 (s)','Mean-based total (h)','Headroom to limit (h)','All tasks at p95 (h)'];table(rt)

*Figure 4. Projected total time for 120 tasks. The table applies the same arithmetic to each configuration's measured mean; the last column assumes that every task is as slow as the 95th percentile, a stress case. A100 times with eight parallel tasks are not the competition's times.*

### 3.4 Public scores

The public leaderboard counts solved tasks out of 58, so one task moves the score by about 0.017 ([analysis by Dariush Afshar](https://www.kaggle.com/code/dariushafshar/gemma-4-lb-58-tasks-bronze-is-a-124-team-tie)). Single public runs gave 0.18 for the unchanged Budget-Fit agent, 0.17 for its issue-contract prompt variant and 0.13 for an alternative prompt; the identical coder–analyzer archive scored 0.12 for its author and 0.10 and 0.06 in later runs. A variant with half the reasoning budget scored 0.17 (`idea_map_leaderboard_observations.csv`). Repeated runs of identical files differ by up to six of the 58 tasks, so single scores cannot rank these configurations; the [companion notebook](https://www.kaggle.com/code/dmitriigluzdov/gemma-4-public-agent-ideas-and-evidence) collects the repeated runs.

In [ ]:
observations=lb[['record_date','configuration','public_score','observation_type','archive_sha256']].copy()
observations.public_score=observations.public_score.map(lambda x:f'{float(x):.2f}')
observations.archive_sha256=observations.archive_sha256.map(lambda x:'Not established' if x=='not_established' else x[:12]+'…')
observations.columns=['Submission / report date (UTC)','Configuration','Public score','Evidence type','Archive SHA-256 prefix'];table(observations)
selected=['Budget-Fit single agent','Calibrated single coder','Contract-first agent','Budget-aware coder','SuperAgent','Direct-code verifier','Coder–analyzer baseline']
public=catalogue[catalogue.configuration.isin(selected)][['configuration','author','best_public_score','license','score_source']].copy()
public.best_public_score=public.best_public_score.map(lambda v:'Not verified' if pd.isna(v) else f'{v:.2f}')
public.score_source=public.score_source.map(lambda u:f'[Source]({u})')
public.columns=['Public notebook','Author','Best public score','License','Evidence'];table(public)

The second table lists public notebooks with their best public score and licence as checked on 13 October 2026; a best score may belong to an older version of a notebook. Catalogue entries are links, not attached code.

## 4. Discussion

The single coder's advantage on development tasks did not carry over to Rich, but its lower time per task did. Quality and time therefore need to be measured together, and a configuration chosen on development tasks needs a check on other repositories; the six-minute time plan is an example that failed it. Extra reviewer calls cost time and can repeat the coder's mistakes, so agreement between them is weak evidence.

Later measurements in the attached datasets point the same way. With the same agent files, the 12B and E4B models solve far fewer tasks than the 31B and are not faster per task, and on the leaderboard the differences between prompt variants stay within run-to-run noise.

## 5. Limitations

The validity check depends on the local test environment. The public tasks come from four repositories, mostly FastAPI, and the hidden tasks may differ. Each comparison uses one seed, and the repeated look at Rich makes it a weaker confirmation. Times measured on an A100 with eight parallel tasks differ from the competition's four L4 GPUs. The serving checks describe the software versions recorded at the time. The adaptation test is small. None of these results estimates the hidden score.

## 6. Conclusion

Valid tasks, faithful serving and a realistic time budget are separate questions, and each needs its own check. Here, a faster agent did not solve more tasks on a second repository, a configuration tuned on development tasks failed its confirmation, and a lower training loss did not mean better repairs. A change is worth adopting only when it solves more tasks in a paired comparison on new repositories and still fits the twelve-hour limit.

## References and credits

1. Jimenez et al. [SWE-bench: Can Language Models Resolve Real-World GitHub Issues?](https://arxiv.org/abs/2310.06770)
2. Hu et al. [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)
3. Google and Kaggle. [Competition data and harness](https://www.kaggle.com/competitions/gemma-4-developer-agent/data), [evaluation](https://www.kaggle.com/competitions/gemma-4-developer-agent/overview/evaluation) and [official starter](https://www.kaggle.com/code/ryanholbrook/getting-started-gemma-4-developer-agent).
4. **verracodeguacas**, [Budget-Fit Single Agent](https://www.kaggle.com/code/verracodeguacas/gemma-4-agent-budget-fit-single-agent), Apache 2.0: the submitted agent.
5. **matterhorn3838**, [SuperAgent](https://www.kaggle.com/code/matterhorn3838/gemma-4-superagent), Apache 2.0: the issue-contract idea behind the prompt variant.
6. **lavinwins**, [Calibrated Single](https://www.kaggle.com/code/lavinwins/gemma-4-autonomous-developer-agent-calibrated-sin), Apache 2.0: a related single-agent design.
7. **Roman Rozen**, [coder–analyzer baseline](https://www.kaggle.com/code/romanrozen/gemma-eda-baseline-for-a-start-lb-top-1), Apache 2.0: the reference agent and its reported score.
8. **Dariush Afshar**, [leaderboard analysis](https://www.kaggle.com/code/dariushafshar/gemma-4-lb-58-tasks-bronze-is-a-124-team-tie): the 58-task public count and repeated runs.
9. KE WU and the organizers, [discussion of the served reasoning field](https://www.kaggle.com/competitions/gemma-4-developer-agent/discussion/744354): the serving mismatch and its correction.

Original measurements and documentation are released under **CC BY 4.0**; utilities and the generated archives under **Apache 2.0**, with the original notices of every agent kept byte for byte.

## Appendix A. Serving checks and a short LoRA test

### A.1 Prompt rendering and adapter effect

In the serving build recorded at the time, the agent framework stored earlier reasoning in a field named `reasoning_content`, while the server read `reasoning`, and plain-text messages were also converted into lists of text parts before the chat template was applied. Applying both conversions reproduces the length of all **803** recorded prompts and all **133** Rich prompts; without them, none of the 803 matches. A live check confirmed identical tokens for **5 of 5** requests. Prompt and completion lengths together match for 765 of 803, so equal prompt length alone does not prove an identical input. The organizers later corrected this behaviour ([discussion](https://www.kaggle.com/competitions/gemma-4-developer-agent/discussion/744354)).

LoRA adds a small trainable update $\Delta W=(\alpha/r)BA$ to frozen weight matrices. To test whether the server applies an adapter at all, an untrained adapter (rank 4, $\alpha$ 8, on 410 projections) was served. Repeated base runs and an adapter with a zero update gave identical outputs, run-to-run drift was **0.125 nats**, the pass threshold was ten times that (**1.25**), and the untrained adapter changed the first-token log-probability by **at least 4.89 nats**. This shows that the adapter path works; it says nothing about solving tasks.

In [ ]:
fig,(ax,bx)=plt.subplots(1,2,figsize=(13,5.5))
r=render.query("cohort == 'native' and check == 'prompt_token_count'")
names={'old_history':'Original history','drop_reasoning_content_only':'Reasoning field only','content_parts_only':'Content parts only','served_history':'Both transformations'}
bars=ax.barh([names[x] for x in r.rendering],100*r.matched/r.total,color=[GRAY,GRAY,AMBER,TEAL]);ax.invert_yaxis()
for bar,row in zip(bars,r.itertuples()):ax.text(bar.get_width()+2,bar.get_y()+bar.get_height()/2,f'{row.matched}/{row.total}',va='center',fontsize=10)
ax.set_xlim(0,126);ax.set_xticks([0,25,50,75,100]);ax.set_xlabel('Recorded prompt lengths reproduced (%)');ax.set_title('Prompt rendering in the recorded build')
keys=['base_repeat_drift','zero_adapter_drift','warmup_drift','effect_threshold','strong_effect_lower_bound']
values=[gate[k] for k in keys];bars=bx.barh(['Base repeat','Zero update','Warmup drift','Effect threshold','Synthetic adapter'],values,color=[GRAY,GRAY,AMBER,BLUE,TEAL]);bx.invert_yaxis()
for i,(bar,v) in enumerate(zip(bars,values)):bx.text(v+.07,bar.get_y()+bar.get_height()/2,('≥ ' if i==4 else '')+f'{v:.3f}',va='center',fontsize=10)
bx.set_xlim(0,6.1);bx.set_xlabel('Change in first-token log-probability (nats)');bx.set_title('Adapter effect against drift')
fig.subplots_adjust(left=.16,right=.99,top=.87,bottom=.25,wspace=.8)
save(fig,'05_serving_diagnostics.png','Figure 5. Prompt rendering and the adapter effect are separate serving checks.','served_prompt_checks.csv; gate_measurements.csv; probe_top20.json')
scope={'__name__':'evidence_replay'};exec(compile(evidence(K,'probe_adapter_effect.py'),'probe_adapter_effect.py','exec'),scope)
replay=scope['assess'](json.loads(evidence(K,'probe_top20.json')))
assert replay['pass'] and replay['missing_token_drop_lower_bound']>replay['threshold']

*Figure 5. Left: share of recorded prompts whose length is reproduced by each rendering (length only, not exact tokens). Right: the untrained adapter's effect against run-to-run drift; the effect is a lower bound.*

### A.2 Lower loss, not more solved tasks

A short training run (rank 16, $\alpha$ 32, learning rate 0.0001, 30 steps on 412 turns from 23 development tasks, sequences up to 8,192 tokens; Rich not used for training) lowered the loss of the model's actions on solved Rich tasks by **40.7%** and of its reasoning by **25.5%**. The loss fell almost as much on unsolved Rich tasks (25.5%, against 27.1% on solved ones), which suggests a general sharpening of the model's predictions rather than better repairs.

In [ ]:
fig,(ax,bx)=plt.subplots(1,2,figsize=(12.4,5.2))
controls=loss.query("span == 'overall' and cohort != 'development_validation'").set_index('cohort')
cohorts=['rich_resolved','rich_unresolved','coder_analyzer_resolved'];v=[-controls.loc[x,'relative_change_pct'] for x in cohorts]
bars=ax.barh(['Rich: resolved','Rich: unresolved','Coder–analyzer: resolved'],v,color=[TEAL,AMBER,BLUE]);ax.invert_yaxis()
for b,value in zip(bars,v):ax.text(value+.5,b.get_y()+b.get_height()/2,f'{value:.1f}%',va='center')
ax.set_xlim(0,35);ax.set_xlabel('Reduction of training loss from base (%)');ax.set_title('Loss falls on solved and unsolved runs alike')
check=adaptation['criteria']['minieval_resolved_not_worse_than_base_minus_1'];n=check['runs'];values=[check['base'],check['adapter']]
bars=bx.bar(['Base','Final adapter'],100*np.array(values)/n,color=[BLUE,TEAL],width=.55);bx.bar_label(bars,labels=[f'{v}/{n}' for v in values],padding=5)
bx.set_ylim(0,20);bx.set_ylabel('Resolved local runs (%)');bx.set_title('Fewer solved runs with the adapter')
fig.subplots_adjust(left=.17,right=.98,top=.87,bottom=.24,wspace=.35)
save(fig,'06_adaptation_controls.png','Figure 6. Lower training loss did not bring more solved tasks.','adaptation_loss_changes.csv; adaptation_evaluation.json')

*Figure 6. On 16 Rich tasks with two seeds and a two-minute limit (32 runs each), the base model solved 4 and the adapter 2, and the final checkpoint's loss on Rich was higher than at mid-training (0.0734 against 0.0678 nats per token). The adapter was therefore not used; 32 runs cannot measure a small difference.*

Serving an adapter, rendering the prompt correctly and solving more tasks are three separate properties, and each needs its own check. The kit dataset now also contains 1,670 trajectories in the competition's format for further LoRA experiments.

## Appendix B. Archives

The notebook writes three archives. **`submission.zip`** is the unchanged public Budget-Fit agent by verracodeguacas, the archive that scored 0.18. It is chosen because its repeated public runs are the most stable among the tested variants: 8 to 11 of 58 tasks over seven runs in Dariush Afshar's analysis. `thinking_2048_variant.zip` halves the reasoning budget from 4,096 to 2,048 tokens (one public run: 0.17), and `issue_contract_prompt_variant.zip` adds an issue checklist to the prompt (one public run: 0.17).

Each archive is rebuilt byte for byte with a fixed file order, timestamps and permissions, and its SHA-256 is checked against the registry in the kit dataset. All three compile with the official packages, keep their original Apache 2.0 notices and involve no model run.

In [ ]:
# Byte-preserved licensed bundle contents.
ISSUE_CONTRACT_FILES={'agent.yaml': 'name: swe_agent\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\ngenerate_content_config: !include configs/sampling.yaml\n', 'configs/sampling.yaml': 'temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 8192\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n', 'eval_config.yaml': 'evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 28\n  max_time_minutes: 8\n  max_turns: 80\n', 'prompts/system.md': 'You are an expert autonomous software engineer assigned to resolve an issue in a repository efficiently and decisively.\n\n## Core Objective: Fast, Minimal, and Precise Fixes\nAim to understand, resolve, and submit the fix in the minimum number of tool calls (under 8–10 turns). Move directly from the problem statement to the relevant files, apply the solution, verify with a targeted test, and submit.\n\n## Workflow\n\n### 1. Identify Target Files Immediately\n- Before the first tool call, extract the issue contract: required names, exception types/messages, return values, defaults and explicitly mentioned edge cases. Keep this checklist short and use it to choose the edit and targeted check; it requires no additional tool call.\n- Extract filenames, functions, classes, CLI subcommands, or error messages directly from the problem statement.\n- Read only the specific target files and lines using `read_file` or search tools. Do not wander across unrelated files.\n- If the problem statement does not provide explicit file paths, search for its most specific identifier or error string with `git grep -n "text" -- \'*.py\' | head -20`.\n- Do not use `search_similar_code`, `get_code_neighbors` or `get_code_subgraph`: they only accept exact fully-qualified ids and usually return nothing.\n\n### 2. Implement the Solution Directly\n- Apply the minimal necessary fix or feature directly to the source files using `edit_file` or `write_file`.\n- Strictly adhere to specified error strings, exception types, HTTP status codes, and API signatures.\n- For a new feature, follow the nearest existing sibling pattern, including required imports, exports or dispatch registration. Reuse the files already localized; keep the edit-by-call-12 deadline and existing verification budget.\n- For documentation code tasks (e.g. FastAPI), edit executable code under `docs_src/`.\n\n### 3. Run Targeted Tests Only (Existing Tests May Be Broken)\n- **Run ONLY Targeted Tests**: Run only the specific test file or test method directly verifying the bug or feature you modified (e.g. `pytest tests/test_target.py -k test_feature` or `python3 -m unittest tests.test_target`).\n- **Be Aware That Existing Tests May Be Broken**: Many repositories contain pre-existing test breakages, missing test data fixtures (e.g. `/test_data`), or environment import errors unrelated to your task.\n- **Do NOT Attempt to Fix Existing Tests**: If an existing test fails due to pre-existing repository issues or missing fixtures, IGNORE IT. Never spend turns attempting to repair pre-existing test failures, create test stubs, or alter test code.\n- **STRICT RULE: NEVER Run Bare Pytest or Full-Repo Sweeps**: NEVER run bare `pytest`, `pytest .`, `python3 -m unittest discover`, or full-repo test suites without specifying a target file. Full test suites take several minutes, cause catastrophic timeouts, and exhaust your turn and time budgets.\n- If you need to locate the test file, find it explicitly with `find tests -name "*<name>*.py"` instead of running the test runner across the repo.\n\n### 4. Immediate Patch Submission\n- Once your targeted test passes:\n  1. Call `submit_patch` immediately.\n  2. Verify `patch_size > 0` and `files_changed > 0`.\n  3. Output a short summary of the fix to end the session.\n\n## Budget: 28 tool calls, 8 minutes\n- You have at most 28 tool calls and 8 minutes for this task. When either runs out, the working tree is graded as it is, so an edit made in time counts even if you never reach `submit_patch`.\n- Use at most about 10 tool calls to find and read the code.\n- By your 12th tool call you must have edited a source file. If you have not, stop exploring and make your best-guess edit in the most likely place now. Runs that are still only reading after 12 calls almost never succeed; more reading does not help.\n- After your edit: one compile check or one targeted test, a correction if needed, then `submit_patch`. Do not start new exploration after your 20th tool call.\n\n## Anti-Patterns to Avoid\n- **NEVER modify, create, or delete test files** (`*_test.py`, `test_*.py`, or anything under `tests/`). All changes must be to source implementation files. Modifying tests results in an automatic evaluation failure.\n- **NEVER run full repository test suites** (e.g., bare `pytest` or `pytest .`) — always specify the exact test file path.\n- **NEVER attempt to fix or repair existing tests or pre-existing repository breakages** — your task is strictly to implement the fix for the reported issue in source code.\n- **NEVER search outside `/workspace`** for source files or packages (e.g., `/usr/local/lib/`, `/wheels/`, `/opt/`). All repository code and test dependencies are pre-installed. If `ModuleNotFoundError` occurs during test runs, focus on fixing code under `/workspace`, not looking for missing system packages.\n- Do NOT spend turns running broad exploratory searches if the file path or symbol is obvious.\n- Do NOT refactor or reformat unrelated functions or files.\n- Do NOT conclude without submitting a non-empty patch (`patch_size > 0`). Every task requires concrete source modifications. Concluding that the codebase is already clean without making changes is an anti-pattern.\n', 'ATTRIBUTION.md': "# D1 attribution and change notice — 2026-10-11\n\nDerived from Wilmer E. Henao (verracodeguacas), Budget-Fit Single Agent:\nhttps://www.kaggle.com/code/verracodeguacas/gemma-4-agent-budget-fit-single-agent\nApache-2.0. The four-file reference was reproduced by our notebook at public 0.18.\n\nTwo short prompt additions by Dmitrii Gluzdov: extract an issue contract without\nanother tool call, and follow sibling registration patterns within existing budgets.\nInspired by matterhorn3838's Apache-2.0 SuperAgent specification/sibling workflow:\nhttps://www.kaggle.com/code/matterhorn3838/gemma-4-superagent\nAll agent YAML, generation settings and evaluation budgets are unchanged.\nD1 has no local quality measurement and no public score. It is ranked below the\nunchanged Budget-Fit reference; compiler success is only compatibility evidence.\n", 'LICENSE.txt': '\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.'}
REFERENCE_FILES={'agent.yaml': 'name: swe_agent\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\ngenerate_content_config: !include configs/sampling.yaml\n', 'configs/sampling.yaml': 'temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 8192\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n', 'eval_config.yaml': 'evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 28\n  max_time_minutes: 8\n  max_turns: 80\n', 'prompts/system.md': 'You are an expert autonomous software engineer assigned to resolve an issue in a repository efficiently and decisively.\n\n## Core Objective: Fast, Minimal, and Precise Fixes\nAim to understand, resolve, and submit the fix in the minimum number of tool calls (under 8–10 turns). Move directly from the problem statement to the relevant files, apply the solution, verify with a targeted test, and submit.\n\n## Workflow\n\n### 1. Identify Target Files Immediately\n- Extract filenames, functions, classes, CLI subcommands, or error messages directly from the problem statement.\n- Read only the specific target files and lines using `read_file` or search tools. Do not wander across unrelated files.\n- If the problem statement does not provide explicit file paths, search for its most specific identifier or error string with `git grep -n "text" -- \'*.py\' | head -20`.\n- Do not use `search_similar_code`, `get_code_neighbors` or `get_code_subgraph`: they only accept exact fully-qualified ids and usually return nothing.\n\n### 2. Implement the Solution Directly\n- Apply the minimal necessary fix or feature directly to the source files using `edit_file` or `write_file`.\n- Strictly adhere to specified error strings, exception types, HTTP status codes, and API signatures.\n- For documentation code tasks (e.g. FastAPI), edit executable code under `docs_src/`.\n\n### 3. Run Targeted Tests Only (Existing Tests May Be Broken)\n- **Run ONLY Targeted Tests**: Run only the specific test file or test method directly verifying the bug or feature you modified (e.g. `pytest tests/test_target.py -k test_feature` or `python3 -m unittest tests.test_target`).\n- **Be Aware That Existing Tests May Be Broken**: Many repositories contain pre-existing test breakages, missing test data fixtures (e.g. `/test_data`), or environment import errors unrelated to your task.\n- **Do NOT Attempt to Fix Existing Tests**: If an existing test fails due to pre-existing repository issues or missing fixtures, IGNORE IT. Never spend turns attempting to repair pre-existing test failures, create test stubs, or alter test code.\n- **STRICT RULE: NEVER Run Bare Pytest or Full-Repo Sweeps**: NEVER run bare `pytest`, `pytest .`, `python3 -m unittest discover`, or full-repo test suites without specifying a target file. Full test suites take several minutes, cause catastrophic timeouts, and exhaust your turn and time budgets.\n- If you need to locate the test file, find it explicitly with `find tests -name "*<name>*.py"` instead of running the test runner across the repo.\n\n### 4. Immediate Patch Submission\n- Once your targeted test passes:\n  1. Call `submit_patch` immediately.\n  2. Verify `patch_size > 0` and `files_changed > 0`.\n  3. Output a short summary of the fix to end the session.\n\n## Budget: 28 tool calls, 8 minutes\n- You have at most 28 tool calls and 8 minutes for this task. When either runs out, the working tree is graded as it is, so an edit made in time counts even if you never reach `submit_patch`.\n- Use at most about 10 tool calls to find and read the code.\n- By your 12th tool call you must have edited a source file. If you have not, stop exploring and make your best-guess edit in the most likely place now. Runs that are still only reading after 12 calls almost never succeed; more reading does not help.\n- After your edit: one compile check or one targeted test, a correction if needed, then `submit_patch`. Do not start new exploration after your 20th tool call.\n\n## Anti-Patterns to Avoid\n- **NEVER modify, create, or delete test files** (`*_test.py`, `test_*.py`, or anything under `tests/`). All changes must be to source implementation files. Modifying tests results in an automatic evaluation failure.\n- **NEVER run full repository test suites** (e.g., bare `pytest` or `pytest .`) — always specify the exact test file path.\n- **NEVER attempt to fix or repair existing tests or pre-existing repository breakages** — your task is strictly to implement the fix for the reported issue in source code.\n- **NEVER search outside `/workspace`** for source files or packages (e.g., `/usr/local/lib/`, `/wheels/`, `/opt/`). All repository code and test dependencies are pre-installed. If `ModuleNotFoundError` occurs during test runs, focus on fixing code under `/workspace`, not looking for missing system packages.\n- Do NOT spend turns running broad exploratory searches if the file path or symbol is obvious.\n- Do NOT refactor or reformat unrelated functions or files.\n- Do NOT conclude without submitting a non-empty patch (`patch_size > 0`). Every task requires concrete source modifications. Concluding that the codebase is already clean without making changes is an anti-pattern.\n'}
THINKING_FILES={'agent.yaml': 'name: swe_agent\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\ngenerate_content_config: !include configs/sampling.yaml\n', 'configs/sampling.yaml': 'temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 8192\nthinking_config:\n  thinking_budget: 2048\n  include_thoughts: true\n', 'eval_config.yaml': 'evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 28\n  max_time_minutes: 8\n  max_turns: 80\n', 'prompts/system.md': 'You are an expert autonomous software engineer assigned to resolve an issue in a repository efficiently and decisively.\n\n## Core Objective: Fast, Minimal, and Precise Fixes\nAim to understand, resolve, and submit the fix in the minimum number of tool calls (under 8–10 turns). Move directly from the problem statement to the relevant files, apply the solution, verify with a targeted test, and submit.\n\n## Workflow\n\n### 1. Identify Target Files Immediately\n- Extract filenames, functions, classes, CLI subcommands, or error messages directly from the problem statement.\n- Read only the specific target files and lines using `read_file` or search tools. Do not wander across unrelated files.\n- If the problem statement does not provide explicit file paths, search for its most specific identifier or error string with `git grep -n "text" -- \'*.py\' | head -20`.\n- Do not use `search_similar_code`, `get_code_neighbors` or `get_code_subgraph`: they only accept exact fully-qualified ids and usually return nothing.\n\n### 2. Implement the Solution Directly\n- Apply the minimal necessary fix or feature directly to the source files using `edit_file` or `write_file`.\n- Strictly adhere to specified error strings, exception types, HTTP status codes, and API signatures.\n- For documentation code tasks (e.g. FastAPI), edit executable code under `docs_src/`.\n\n### 3. Run Targeted Tests Only (Existing Tests May Be Broken)\n- **Run ONLY Targeted Tests**: Run only the specific test file or test method directly verifying the bug or feature you modified (e.g. `pytest tests/test_target.py -k test_feature` or `python3 -m unittest tests.test_target`).\n- **Be Aware That Existing Tests May Be Broken**: Many repositories contain pre-existing test breakages, missing test data fixtures (e.g. `/test_data`), or environment import errors unrelated to your task.\n- **Do NOT Attempt to Fix Existing Tests**: If an existing test fails due to pre-existing repository issues or missing fixtures, IGNORE IT. Never spend turns attempting to repair pre-existing test failures, create test stubs, or alter test code.\n- **STRICT RULE: NEVER Run Bare Pytest or Full-Repo Sweeps**: NEVER run bare `pytest`, `pytest .`, `python3 -m unittest discover`, or full-repo test suites without specifying a target file. Full test suites take several minutes, cause catastrophic timeouts, and exhaust your turn and time budgets.\n- If you need to locate the test file, find it explicitly with `find tests -name "*<name>*.py"` instead of running the test runner across the repo.\n\n### 4. Immediate Patch Submission\n- Once your targeted test passes:\n  1. Call `submit_patch` immediately.\n  2. Verify `patch_size > 0` and `files_changed > 0`.\n  3. Output a short summary of the fix to end the session.\n\n## Budget: 28 tool calls, 8 minutes\n- You have at most 28 tool calls and 8 minutes for this task. When either runs out, the working tree is graded as it is, so an edit made in time counts even if you never reach `submit_patch`.\n- Use at most about 10 tool calls to find and read the code.\n- By your 12th tool call you must have edited a source file. If you have not, stop exploring and make your best-guess edit in the most likely place now. Runs that are still only reading after 12 calls almost never succeed; more reading does not help.\n- After your edit: one compile check or one targeted test, a correction if needed, then `submit_patch`. Do not start new exploration after your 20th tool call.\n\n## Anti-Patterns to Avoid\n- **NEVER modify, create, or delete test files** (`*_test.py`, `test_*.py`, or anything under `tests/`). All changes must be to source implementation files. Modifying tests results in an automatic evaluation failure.\n- **NEVER run full repository test suites** (e.g., bare `pytest` or `pytest .`) — always specify the exact test file path.\n- **NEVER attempt to fix or repair existing tests or pre-existing repository breakages** — your task is strictly to implement the fix for the reported issue in source code.\n- **NEVER search outside `/workspace`** for source files or packages (e.g., `/usr/local/lib/`, `/wheels/`, `/opt/`). All repository code and test dependencies are pre-installed. If `ModuleNotFoundError` occurs during test runs, focus on fixing code under `/workspace`, not looking for missing system packages.\n- Do NOT spend turns running broad exploratory searches if the file path or symbol is obvious.\n- Do NOT refactor or reformat unrelated functions or files.\n- Do NOT conclude without submitting a non-empty patch (`patch_size > 0`). Every task requires concrete source modifications. Concluding that the codebase is already clean without making changes is an anti-pattern.\n'}

def build_archive(files,record):
    path=WORK/record['output_file']
    with zipfile.ZipFile(path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=9) as z:
        for name in sorted(files):
            assert not name.startswith(('/','\\')) and '..' not in Path(name).parts
            item=zipfile.ZipInfo(name,date_time=(1980,1,1,0,0,0));item.create_system=3;item.external_attr=0o644<<16
            z.writestr(item,files[name].encode('utf8'),compress_type=zipfile.ZIP_DEFLATED,compresslevel=9)
    digest=hashlib.sha256(path.read_bytes()).hexdigest();assert digest==record['archive_sha256']
    with zipfile.ZipFile(path) as z:assert z.testzip() is None and z.namelist().count('agent.yaml')==1
    return {'Archive':path.name,'Configuration':record['configuration'],'SHA-256':digest,'License':record['license']}
assert all(ISSUE_CONTRACT_FILES[k]==REFERENCE_FILES[k] for k in ['agent.yaml','configs/sampling.yaml','eval_config.yaml'])
assert [k for k in THINKING_FILES if THINKING_FILES[k]!=REFERENCE_FILES[k]]==['configs/sampling.yaml']
assert THINKING_FILES['configs/sampling.yaml']==REFERENCE_FILES['configs/sampling.yaml'].replace('thinking_budget: 4096','thinking_budget: 2048')
assert registry[1]['configuration']=='Budget-Fit single agent'
archives=[build_archive(REFERENCE_FILES,{**registry[1],'output_file':'submission.zip'}),
          build_archive(THINKING_FILES,{**registry[0],'output_file':'thinking_2048_variant.zip'}),
          build_archive(ISSUE_CONTRACT_FILES,registry[2])]
(WORK/'archive_manifest.json').write_text(json.dumps(archives,indent=2)+'\n',encoding='utf8')
(WORK/'BUNDLE_LICENSE.txt').write_text(ISSUE_CONTRACT_FILES['LICENSE.txt'],encoding='utf8')
credits=json.loads(evidence(K,'credits.json'))
(WORK/'BUNDLE_CREDITS.md').write_text('# Attribution\n\n'+'\n'.join(f"- {c['name']} by {c['author']} ({c['license']}): {c['url']} — {c['contribution']}" for c in credits)+'\n',encoding='utf8')
table(pd.DataFrame(archives))